In [0]:
from pyspark.sql.functions import *
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.window import Window

In [0]:
mds = spark.read.table("hive_metastore.default.mds")
weather = spark.read.table("hive_metastore.default.weather_gold")
df = spark.read.table("hive_metastore.default.model_table")

1- Juntar mds ao df
2- Juntar weather ao df

In [0]:
display(weather)

In [0]:
display(df)

In [0]:
df.select("ID").distinct().count()


In [0]:
# Total number of distinct objects
total_objects = df.select("ID").distinct().count()

# Objects with TIME_OVER_LIMIT_T > 0 or TIME_OVER_LIMIT_I > 0
objects_over_limit = df.filter(
    (col("TIME_OVER_LIMIT_T") > 0) | (col("TIME_OVER_LIMIT_I") > 0)
).select("ID").distinct().count()

# Calculate percentage
percentage = (objects_over_limit / total_objects) * 100

print(f"Percentage of objects over limit: {percentage:.2f}%")

In [0]:
display(mds)

In [0]:
test = mds.filter(col("TAGCOM").contains("NP"))
display(test)

In [0]:
# Get distinct IDs from df and first 6 chars
df_ids = df.select(substring("ID", 1, 6).alias("match_key")).distinct()

# Get distinct TAGCOMs from mds and first 6 chars
mds_keys = mds.select(substring("TAGCOM", 1, 6).alias("match_key")).distinct()

# Anti-join to find IDs in df not matched in mds
unmatched = df_ids.join(mds_keys, on="match_key", how="left_anti")

# Count unmatched
unmatched_count = unmatched.count()

if unmatched_count == 0:
    print("✅ All IDs in df have a matching TAGCOM in mds (first 6 characters).")
else:
    print(f"⚠️ {unmatched_count} ID(s) in df do not have a matching TAGCOM in mds.")
    display(unmatched)

In [0]:
mds = mds.select("ID_OBJECTO","CONCELHO","AO","NOME","TIPOINST","TAGCOM","REDE","CHAVE_SIT","X_SIT","Y_SIT","MODELO")

In [0]:
df_mds = df.join(
    mds,
    substring(df["ID"], 1, 6) == substring(mds["TAGCOM"], 1, 6),
    how="inner"
)

display(df_mds)

In [0]:
test = df_mds.select("REDE").distinct()
display(test)

In [0]:
test = df_mds.select("ID", "TIPOINST").distinct().filter(col("ID").contains("OSPAAA"))
display(test)

In [0]:
display(df_mds.filter(col("ID").contains("OSPAAA")))

In [0]:
df_mds.select("ID").distinct().count()


In [0]:
df_prefix_counts = df.select(
    substring("ID", 1, 9).alias("id_prefix"),
    "ID"
).distinct().groupBy("id_prefix").agg(countDistinct("ID").alias("distinct_ids"))

# Filter prefixes that map to more than 1 ID
ambiguous_prefixes = df_prefix_counts.filter("distinct_ids > 1")

display(ambiguous_prefixes)

Juntar tempo aos dados

In [0]:
weather = weather.withColumnRenamed("date", "weather_date")

In [0]:
df_mds = df_mds.withColumn("day", to_date("date"))
weather_df = weather.withColumn("day", to_date("weather_date"))

In [0]:
display(df_mds)

In [0]:
# Drop duplicates to get one row per object
df_objects = df_mds.select("ID", "X_SIT", "Y_SIT").dropDuplicates()

# Cross with weather station list (distinct lat/lon)
stations = weather_df.select("location", "latitude", "longitude").dropDuplicates()

# Join every object with every station
df_cross_static = df_objects.crossJoin(stations)

# Compute distance
df_cross_static = df_cross_static.withColumn(
    "dist",
    sqrt(pow(col("X_SIT") - col("longitude"), 2) + pow(col("Y_SIT") - col("latitude"), 2))
)

# Get closest station per object
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("ID").orderBy("dist")
df_station_map = df_cross_static.withColumn("row", row_number().over(w)) \
                                .filter("row = 1") \
                                .select("ID", "location")

In [0]:
df_mds_with_station = df_mds.join(df_station_map, on="ID", how="left")

In [0]:
df_mds_with_station = df_mds_with_station.withColumn("day", to_date("date"))
weather_df = weather_df.withColumn("day", to_date("weather_date"))  # or to_date("DATE")

df_final = df_mds_with_station.join(weather_df, on=["location", "day"], how="left")

In [0]:
display(df_final)

In [0]:
display(df_final.filter(col("ID") == "APACL-5502-0"))

In [0]:
df_final = df_final.drop("day","weather_date","location","latitude","longitude","X_SIT","Y_SIT","MODELO","DATE_ONLY","CHAVE_SIT")
display(df_final)

In [0]:
display(df.filter(col("ID") == "APACL-5502-0"))

In [0]:


permanent_table_name = "model_table_weather"

df_final.write.format("parquet").saveAsTable(permanent_table_name)